In [0]:
dim_billing_type = spark.sql(f"select * from regis_healthcare.silver.billing;")
dim_billing_type.createOrReplaceTempView("billing")

In [0]:
# 7. Dim_Billing_Type
# | Column           |
# | ---------------- |
# | billing_type_key |
# | billing_type     |
# display(df_billing)
#----------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dim_billing_type = spark.sql("""
    select distinct(billing_type) from billing""")

dim_billing_type.createOrReplaceTempView("bill")

dim_billing_type = spark.sql("""
    select billing_type,row_number()over(order by billing_type )as billing_type_key from bill group by billing_type """)
#----------------
dim_billing_type = dim_billing_type.select(
"billing_type_key",
"billing_type")
display(dim_billing_type)


#### cataloge 

In [0]:
# dim_billing_type.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.dim_billing_type")

In [0]:
dim_billing_type.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_billing_type")
print(dim_billing_type.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_billing_type")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_billing_type")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.billing_type_key = source.billing_type_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_billing_type;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_billing_type;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# # gold load to s3
# dim_billing_type.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/dim_billing_type")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_billing_type"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = dim_billing_type

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.billing_type_key = source.billing_type_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
